# PV181 RNG

This notebook contains python code for several tasks treated in this seminar. 

# PRNG 

## Determinism of PRNG

We will work with PRNG implemented in [random](https://docs.python.org/3/library/random.html) package. See first 4 methods (`seed, setstate, getstate, randbytes`) in the documentation. 

**Task 1**. Import **random** package. </br>
**Task 2**. Generate (and print) 10 random bytes.</br>
**Task 3**. Print out bytes in hexadecimal form (use `.hex()` method). Execute cell 2x. PRNG produces different results since generation is non-deterministic when it is seeded by time.</br>
**Task 4**. Use fixed `seed` and verify that generation is deterministic (generated bytes are always the same).</br>

In [ ]:
import random

def random_bytes(n):
    # random.randbytes was added in Python 3.9.
    if hasattr(random, 'randbytes'):
        return random.randbytes(n)
    return random.getrandbits(n * 8).to_bytes(n, 'little')

rnd_bytes = random_bytes(10)
print(rnd_bytes)
print(rnd_bytes.hex())

random.seed(1)
first = random_bytes(10)
random.seed(1)
second = random_bytes(10)
assert first == second
print('Fixed-seed output:', first.hex())

**Task 5**. Verify that seed determines internal state. For `seed = 1`, what is the internal state?<br/> (Check format: `(3, (2147483648, ...))`?)</br>
**Task 6**. Set the PRNG to that state (instead of using seed) and generate 10 bytes. You should get the same bytes as step 5.</br>

In [ ]:
random.seed(1)
state = random.getstate()
from_seed = random_bytes(10)
random.setstate(state)
from_state = random_bytes(10)
assert from_seed == from_state
print('State:', state)
print('Bytes:', from_state.hex())

**Task 7**. **Attack**: The generator produced 16 bytes that will be used as AES key. The first half of the key is `73a9bef499bbf4dc`. Find the rest of the key by trying seeds from `0` through `9`.<br/><br/>   **Hint**: user used a small seed.

In [ ]:
for seed in range(10):
    random.seed(seed)
    K = random_bytes(16)
    print(K.hex())

**Task 8**. **Attack (time-based seed):**
- **Part A (Generate):** Create random bytes using time as seed, record the bytes
- **Part B (Attack):** Brute-force to recover the seed from the bytes you just generated

In [ ]:
import random, time, datetime

# PART A: Generate random bytes using current time as seed - just execute
t_generate = int(time.time())
random.seed(t_generate)
rnd_bytes = random_bytes(10)

print("Bytes generated:", rnd_bytes.hex())
print("Time:", datetime.datetime.fromtimestamp(t_generate).isoformat())

In [ ]:
# PART B: Brute-force to recover the seed
for s in range(t_generate - 60, t_generate + 61):
    random.seed(s)
    if random_bytes(10) == rnd_bytes:
        print(f"Found! Seed: {s}")

# LCG
Standard PRNG functions are fast but **insecure**. They leak internal state: one generated value is enough to regenerate all future values. LCG is even weaker—you can also reverse it to find previous values.

- **Python's random module** uses [Mersenne Twister](https://en.wikipedia.org/wiki/Mersenne_Twister) (state: 625 32-bit values)
- **Other languages** (C, Java) typically use LCG with state updated as: $$state = (state*a+c) \pmod m$$
- <span style="color:red">**Key fact:** In LCG, the state itself is the random value returned!</span>

## LCG (ANSI C)

Here's ANSI C's LCG implementation (from the C standard):

```c
static unsigned long int next = 1;
void srand(unsigned int seed) { next = seed; }
int rand(void) { return next = (next * 1103515245 + 12345) & 0x7fffffff; }
```

**Task 9**. Create a Python `class PRNG` implementing this LCG.</br>
**Task 10**. Generate 10 values with `seed=0`. First value should be `12345`.</br>
**Task 11**. Find the seed that produces sequence starting with `1406932606`.

In [ ]:
class LCG:
    def __init__(self, a, c, m):
        self.a = a
        self.c = c
        self.m = m
        self.srand(0)

    def srand(self, seed):
        self.state = seed

    def rand(self):
        self.state = (self.state * self.a + self.c) % self.m
        return self.state
        

ansi_rand = LCG(a=1103515245, c=12345, m=2**31)
ansi_rand.srand(0)
rnd_values = [ansi_rand.rand() for i in range(10)]
print(rnd_values)
assert rnd_values[:2] == [12345, 1406932606]

### LCG is reversible (Advanced)

**Task 12**. Find the two missing states `??` in the sequence `[??, ??, 12345, 1406932606]` by reversing the LCG.



LCGs are reversible because every operation (multiply by a, add c, mod m) 
has an inverse. Derive the backward equation from the forward one:

$$state_{new} = (state_{old} \cdot a + c) \bmod m$$
$$state_{old} = (state_{new} \cdot a_{back} + c_{back}) \bmod m$$

where `a_back` and `c_back` are the reverse coefficients computed from `a, c`. If stuck, check the solution in **PV181_RNG_python_solution.ipynb**. <br/><br/>**Hint:**  Use modular inverse: `pow(a, -1, m)` to reverse the LCG.



In [ ]:
a = 1103515245
c = 12345
m = 2**31
a_back = pow(a, -1, m)
c_back = (-c * a_back) % m
ansi_rand_backward = LCG(a_back, c_back, m)
ansi_rand_backward.srand(1406932606)
backward_sequence = [ansi_rand_backward.rand() for _ in range(3)]
print("Reverse sequence:", backward_sequence)
print("Missing states in forward order:", [2088216195, 0])

### LCG (Java random)

Execute this command to generate Java's `java.util.Random` sequence:

```
python3 codes/LCG.py -m 2**48 -a 25214903917 -c 11 -s 1 -l 16 -u 47 -n 10
```

Parameters: `-m` modulus, `-a` multiplier, `-c` increment, `-s` seed, `-l` lowest bit, `-u` highest bit, `-n` values generated

**Task 13**. Verify your LCG implementation against Java's output.

# Small state attack

This generator uses cryptographic hash functions (`SHA1`, `SHA256`) but has a fatal flaw: **small state** (1 Byte = 256 possible values). Although hash functions are one-way, we can brute-force all 256 states.

**Task 14**. Execute the cell below and explore the generator. Find the period (how many values before it repeats).

<span style="color:blue">**Interesting fact:** Some values are not repeated. Why? There is a pre-period due to improperly designed generator. Hash functions are random functions, not linear, hence they do not produce cycles but repetition starts from some point, not from the beginning like LCG.</span>

In [ ]:
from cryptography.hazmat.primitives import hashes
import os

def SHA1(message: bytes):
    digest = hashes.Hash(hashes.SHA1())
    digest.update(message)
    return digest.finalize() 

def SHA256(message: bytes):
    digest = hashes.Hash(hashes.SHA256())
    digest.update(message)
    return digest.finalize() 

class PRNG:
    def __init__(self, state_size = 1):
        self.state_size = state_size
        self.srand(os.urandom(16))

    def srand(self, seed):
        self.state = SHA256(seed)[:self.state_size]

    def rand_bytes(self, num_bytes=10):
        rnd = SHA256(self.state)[:num_bytes]
        self.state = SHA1(self.state)[:self.state_size]
        return bytes(rnd)

rng = PRNG()
sequence = [rng.rand_bytes(5).hex() for i in range(30)]
print(sequence)

# Detect the preperiod and cycle of the one-byte state transition.
rng = PRNG()
seen = {}
states = []
while rng.state not in seen:
    seen[rng.state] = len(states)
    states.append(rng.state)
    rng.rand_bytes(5)
cycle_start = seen[rng.state]
print('Preperiod:', cycle_start)
print('Cycle length:', len(states) - cycle_start)

# <span style="color:blue">Interesting fact: Some values are not repeated. Why? There is a pre-period due to improperly designed generator. Hash functions are random functions, not linear, hence they do not produce cycles but repetition starts from some point, not from the beginning like LCG.</span>

**Task 15**. **Attack**: Find all possible 16 byte blocks `rng` can produce.

In [ ]:
all_keys = []
for i in range(256):
    rng.state = bytes([i])
    all_keys.append(rng.rand_bytes(16))

**Task 16**. The generator was used to create random key `K1`. Message b'arbitrarymessage' was encrypted with `K1` to produce ciphertext `CT1`. Find the key `K1` by brute-forcing all possible keys and checking which one decrypts to normal text.

In [ ]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes


def encrypt_ECB(key, msg):
    cipher = Cipher(algorithms.AES(key), modes.ECB())
    enc = cipher.encryptor()
    ct = enc.update(msg) + enc.finalize()
    return ct 

def decrypt_ECB(key, ct):
    cipher = Cipher(algorithms.AES(key),  modes.ECB())
    dec = cipher.decryptor()
    pt = dec.update(ct) + dec.finalize()
    return pt

rng = PRNG()
CT1 = encrypt_ECB(rng.rand_bytes(16), b'arbitrarymessage')
CT2 = b'\xb0\xc5\x1f5\x87,JH2\xd1\\8\xb0\xd4-Y'

In [ ]:
# Task 16: Brute-force to find K1
for key in all_keys:
    if decrypt_ECB(key, CT1) == b'arbitrarymessage':
        print('K1:', key.hex())

**Task 17**. Another key `K2` was used to encrypt unknown message `PT2` to produce `CT2`. Brute-force to find `K2`, then decrypt `CT2` to recover `PT2`. Use the known plaintext pattern (printable ASCII text) to identify the correct key.

In [ ]:
# Task 17: Find K2 and decrypt CT2
for key in all_keys:
    plaintext = decrypt_ECB(key, CT2)
    if all(32 <= byte <= 126 for byte in plaintext):
        print('K2:', key.hex())
        print('PT2:', plaintext)

# TRNG

## Sources: dev/random, dev/urandom
These files provide **cryptographically secure** random bytes from the OS entropy pool. Unlike PRNGs, they cannot be predicted.

Reading random bytes from dev/urandom:

In [ ]:
import os
import secrets 
os.urandom(10)
print(secrets.token_bytes(10).hex())
print(os.getrandom(10).hex())

cff71a8f67e858876af1
767d16dabe6d5eaf0b41


Files **dev/random**, **dev/urandom** can be also opened as binary file for reading.  
Then you can read specified number of bytes e.g. 10. 

In [ ]:
with open("/dev/urandom", "rb") as random_source:
    print(random_source.read(10).hex())

b'\xfd\x05\xd5\xc1k\x888\x1b\xc7\x00'

# Testing correlation of bits

**Task 18**. Implement function `histogram(rnd_bytes, i, j)` that extracts bits at positions `i` and `j` from each byte and counts the frequencies of all 4 possible 2-bit patterns (00, 01, 10, 11). Returns dictionary with 4 frequency counts.

In [ ]:
# Task 18: Implement function histogram(rnd_bytes, i, j)
import os

def histogram(rnd_bytes, i, j):
    if i == j or not (0 <= i < 8 and 0 <= j < 8):
        raise ValueError("i and j must be distinct bit positions in 0..7")
    mask = (1 << i) | (1 << j)
    hist = {0: 0, 1 << i: 0, 1 << j: 0, mask: 0}
    for byte in rnd_bytes:
        hist[byte & mask] += 1
    return hist

test_bytes = bytes([0, 1, 2, 3, 0, 1, 2, 3])
result = histogram(test_bytes, 0, 1)
print("Test histogram(test_bytes, 0, 1):", result)
assert result == {0: 2, 1: 2, 2: 2, 3: 2}

**Task 19**. Verify that for arbitrary params `i,j` and size of generated block the frequencies are roughly equal.

In [ ]:
# Task 19: Explore TRNG uniformity
# Verify that for arbitrary params i,j and size of generated block
# the frequencies are roughly equal (uniformity test)

rnd_bytes = os.urandom(1000)
for i, j in [(0, 1), (2, 5), (6, 7)]:
    result = histogram(rnd_bytes, i, j)
    print(f"Pair ({i}, {j}):", result)
    # Expected: all 4 frequencies roughly equal (~250 each, ±50 variation)

# Testing correlation of bits

**Task 18**. Implement function `histogram(rnd_bytes, i, j)` that extracts bits at positions `i` and `j` from each byte and counts the frequencies of all 4 possible 2-bit patterns (00, 01, 10, 11). Returns dictionary with 4 frequency counts.

---

**Task 19**. Verify that for arbitrary params `i,j` and size of generated block the frequencies are roughly equal.

---

**Task 20**. Generate 1000 random bytes from `ansi_rand` (LCG from earlier section). Then explore bit correlations: use the histogram function to test multiple bit pairs (i,j). Look for anomalies—**Which bit pairs deviate from uniform distribution?** Compare your LCG results with TRNG results from Task 19.

Find params `i,j` where all frequencies are exactly the same. <span style="color:red">**Generator with such perfect results is also problematic!**</span> **Can we predict some (next) bits with better probability than 50%?** This reveals the LCG's underlying weakness—certain bit positions are interdependent, breaking the randomness assumption.

In [ ]:
# Task 20: Generate LCG bytes and explore bit correlations
ansi_rand.srand(0)
lcg_bytes = bytes([ansi_rand.rand() % 256 for _ in range(1000)])

for i, j in [(0, 1), (2, 5), (6, 7)]:
    print((i, j), histogram(lcg_bytes, i, j))

# Find params i,j where all frequencies are exactly the same
# <span style="color:red">Generator with such perfect results is also problematic!</span>
# Can we predict some (next) bits with better probability than 50%?